# MICE 補完可視化ノートブック

`spot_rate_prediction_v1.ipynb` で使用する MICE 補完の品質を確認する。

## 確認ポイント
1. 補完箇所（橙色）が前後の値と自然につながっているか
2. 補完値がスパイク（異常値）になっていないか
3. テナー間で補完値に矛盾がないか（例: 3M < 6M < 1Y の関係が崩れていないか）

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

from data.utils.database_manager import DatabaseManager

pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup complete.')

## 1. データ読み込み

In [ ]:
db = DatabaseManager()

TARGET_TENORS = [
    '3M', '6M', '9M', '1Y',
    '2Y', '3Y', '4Y', '5Y', '6Y', '7Y', '8Y', '9Y',
    '10Y', '11Y', '12Y', '15Y', '20Y', '25Y', '30Y', '35Y', '40Y'
]
DATA_START = '2014-06-23'

# OIS データ
tenor_list = "', '".join(TARGET_TENORS)
ois_rows = db.select_as_dict(f"""
    SELECT trade_date, tenor, rate
    FROM irs_data
    WHERE product_type = 'OIS'
      AND tenor IN ('{tenor_list}')
      AND trade_date >= '{DATA_START}'
    ORDER BY trade_date, tenor
""")
df_ois_long = pd.DataFrame(ois_rows)
df_ois_long['trade_date'] = pd.to_datetime(df_ois_long['trade_date'])
df_ois_long['rate'] = pd.to_numeric(df_ois_long['rate'], errors='coerce').astype(float)

df_ois = df_ois_long.pivot(index='trade_date', columns='tenor', values='rate')
df_ois = df_ois.reindex(columns=TARGET_TENORS)
df_ois.columns = [f'OIS_{c}' for c in df_ois.columns]

# アンカー列
fx_rows = db.select_as_dict(f"SELECT trade_date, currency_pair, close_price FROM exchange_rates WHERE currency_pair IN ('USDJPY', 'DXY') AND trade_date >= '{DATA_START}'")
df_fx = pd.DataFrame(fx_rows)
df_fx['trade_date'] = pd.to_datetime(df_fx['trade_date'])
df_fx['close_price'] = pd.to_numeric(df_fx['close_price'], errors='coerce').astype(float)
df_fx = df_fx.pivot(index='trade_date', columns='currency_pair', values='close_price')

ust_rows = db.select_as_dict(f"SELECT trade_date, tenor, yield_value FROM foreign_yields WHERE region='US' AND instrument_type='sovereign' AND tenor IN ('2Y','5Y','10Y','30Y') AND trade_date >= '{DATA_START}'")
df_ust = pd.DataFrame(ust_rows)
df_ust['trade_date'] = pd.to_datetime(df_ust['trade_date'])
df_ust['yield_value'] = pd.to_numeric(df_ust['yield_value'], errors='coerce').astype(float)
df_ust = df_ust.pivot(index='trade_date', columns='tenor', values='yield_value')
df_ust.columns = [f'UST_{c}' for c in df_ust.columns]

nk_rows = db.select_as_dict(f"SELECT trade_date, close_price as nikkei FROM stock_prices WHERE ticker='^N225' AND trade_date >= '{DATA_START}'")
df_nk = pd.DataFrame(nk_rows)
df_nk['trade_date'] = pd.to_datetime(df_nk['trade_date'])
df_nk['nikkei'] = pd.to_numeric(df_nk['nikkei'], errors='coerce').astype(float)
df_nk = df_nk.set_index('trade_date')

ffr_rows = db.select_as_dict(f"SELECT trade_date, yield_value as ffr FROM foreign_yields WHERE region='US' AND instrument_type='policy' AND tenor='ON' AND trade_date >= '{DATA_START}'")
df_ffr = pd.DataFrame(ffr_rows)
df_ffr['trade_date'] = pd.to_datetime(df_ffr['trade_date'])
df_ffr['ffr'] = pd.to_numeric(df_ffr['ffr'], errors='coerce').astype(float)
df_ffr = df_ffr.set_index('trade_date')

# 結合
df = df_ois.join([df_fx, df_ust, df_nk, df_ffr], how='left')
ois_raw_cols = [c for c in df_ois.columns]
anchor_cols = [c for c in ['UST_2Y','UST_5Y','UST_10Y','UST_30Y','ffr','USDJPY','DXY','nikkei'] if c in df.columns]

print(f'データ形状: {df.shape}')
print(f'OIS 列: {len(ois_raw_cols)}')
print(f'アンカー列: {anchor_cols}')
print()
print('=== 欠損状況 ===')
missing = df[ois_raw_cols].isna().sum()
print(missing[missing > 0])

## 2. MICE 補完実行（本番と同一設定）

In [ ]:
# 補完前の原本を保存
df_orig = df[ois_raw_cols].copy()

# 補完前フラグ
for col in ois_raw_cols:
    df[f'{col}_is_imputed'] = df[col].isna().astype(int)

total_missing = sum(df[f'{col}_is_imputed'].sum() for col in ois_raw_cols)
print(f'補完が必要なセル総数: {total_missing}')

# MICE 実行（spot_rate_prediction_v1.ipynb と同一設定）
impute_target_cols = ois_raw_cols + anchor_cols

imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42,
    imputation_order='ascending',   # 欠損の少ない列（アンカー）から順に補完
    verbose=0,
)

print('MICE 実行中... (しばらくかかります)')
imputed_matrix = imputer.fit_transform(df[impute_target_cols])
imputed_df = pd.DataFrame(imputed_matrix, index=df.index, columns=impute_target_cols)

# OIS 列のみ書き戻し
df[ois_raw_cols] = imputed_df[ois_raw_cols]

print('MICE 完了')
print(f'補完後の欠損: {df[ois_raw_cols].isna().sum().sum()}')

## 3. 補完品質の可視化

### 3-1. 欠損の多い短期テナー（3M, 6M, 9M）

In [ ]:
def plot_imputation(df_imputed, df_original, col, ax, last_n=400):
    """補完前後を比較プロット。橙色点が補完箇所。"""
    imputed_mask = df_imputed[f'{col}_is_imputed'] == 1
    n_imputed = imputed_mask.sum()

    # 直近 last_n 件のみ表示
    tail_idx = df_imputed.index[-last_n:]
    orig_tail = df_original.loc[tail_idx, col]
    imp_tail  = df_imputed.loc[tail_idx, col]
    mask_tail = imputed_mask.loc[tail_idx]

    ax.plot(tail_idx, imp_tail, color='steelblue', lw=1, label='補完済み', alpha=0.9)
    if mask_tail.sum() > 0:
        ax.scatter(tail_idx[mask_tail], imp_tail[mask_tail],
                   color='orange', s=15, zorder=5, label=f'補完点 ({n_imputed}件)')
    ax.set_title(f'{col.replace("OIS_","OIS ")} (補完点: {n_imputed}件)', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

# 短期テナー可視化
short_tenors = ['OIS_3M', 'OIS_6M', 'OIS_9M', 'OIS_1Y']
fig, axes = plt.subplots(len(short_tenors), 1, figsize=(14, 10), sharex=True)
fig.suptitle('MICE 補完品質確認 - 短期テナー（直近400日）', fontsize=13)

for ax, col in zip(axes, short_tenors):
    if col in df.columns:
        plot_imputation(df, df_orig, col, ax)

plt.tight_layout()
plt.savefig('../../analysis/ml/mice_check_short_tenors.png', dpi=120, bbox_inches='tight')
plt.show()

### 3-2. 中長期テナー（代表）

In [ ]:
mid_long_tenors = ['OIS_2Y', 'OIS_5Y', 'OIS_10Y', 'OIS_20Y', 'OIS_30Y', 'OIS_40Y']
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('MICE 補完品質確認 - 中長期テナー（直近400日）', fontsize=13)

for ax, col in zip(axes.flatten(), mid_long_tenors):
    if col in df.columns:
        plot_imputation(df, df_orig, col, ax)

plt.tight_layout()
plt.savefig('../../analysis/ml/mice_check_mid_long_tenors.png', dpi=120, bbox_inches='tight')
plt.show()

### 3-3. カーブ形状チェック（補完後に逆転が起きていないか）

In [ ]:
# 補完箇所での OIS カーブ形状を確認
# 3M < 6M < 1Y の順序が崩れている日はあるか
short_check = df[['OIS_3M', 'OIS_6M', 'OIS_9M', 'OIS_1Y']].copy()
short_check['3M_6M_ok'] = short_check['OIS_3M'] <= short_check['OIS_6M']
short_check['6M_9M_ok'] = short_check['OIS_6M'] <= short_check['OIS_9M']
short_check['9M_1Y_ok'] = short_check['OIS_9M'] <= short_check['OIS_1Y']

violations = short_check[~(short_check['3M_6M_ok'] & short_check['6M_9M_ok'] & short_check['9M_1Y_ok'])]
print(f'カーブ逆転が発生している日数: {len(violations)} / {len(short_check)}')
if len(violations) > 0:
    print('(サンプル):')
    print(violations[['OIS_3M','OIS_6M','OIS_9M','OIS_1Y']].head(10))
else:
    print('逆転なし (補完品質良好)')

# 補完点でのスポットレート水準の妥当性チェック
print()
print('=== 補完値の統計 vs 非補完値の統計 (OIS_3M) ===')
if 'OIS_3M_is_imputed' in df.columns and df['OIS_3M_is_imputed'].sum() > 0:
    imputed_vals   = df.loc[df['OIS_3M_is_imputed'] == 1, 'OIS_3M']
    original_vals  = df.loc[df['OIS_3M_is_imputed'] == 0, 'OIS_3M']
    print(f'  補完値:   mean={imputed_vals.mean():.4f}, std={imputed_vals.std():.4f}, min={imputed_vals.min():.4f}, max={imputed_vals.max():.4f}')
    print(f'  非補完値: mean={original_vals.mean():.4f}, std={original_vals.std():.4f}, min={original_vals.min():.4f}, max={original_vals.max():.4f}')
else:
    print('OIS_3M に補完箇所なし')

### 3-4. 全テナーの欠損・補完サマリー

In [ ]:
summary = []
for col in ois_raw_cols:
    flag_col = f'{col}_is_imputed'
    n_missing = df[flag_col].sum() if flag_col in df.columns else 0
    summary.append({
        'Tenor': col.replace('OIS_', ''),
        'Total_rows': len(df),
        'Imputed': n_missing,
        'Imputed_%': round(n_missing / len(df) * 100, 2),
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

# 可視化
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df_summary['Tenor'], df_summary['Imputed_%'], color='steelblue', alpha=0.8)
ax.set_xlabel('Tenor')
ax.set_ylabel('補完率 (%)')
ax.set_title('テナー別 MICE 補完率')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../../analysis/ml/mice_imputation_rate.png', dpi=120, bbox_inches='tight')
plt.show()